# Building simple UC tools
We'll create a few functions to help your agent:
1. compute high‑value at‑risk customers.

2. get customer profile information.

3. get customer purchase history.

4. list available products for a given franchise, so the agent doesn't hallucinate items. 
## 1/ Compute high‑value at‑risk customers
Let's define a function to get high-value customers as a table-valued function. High value customers are defined as those who have spent more than the average amount spent by all customers in the last 14 days.
- The function takes two arguments: top_k and drop_window_days with default values of 10 and 14 respectively.
- The function returns a table with three columns: customer_id, total_spent, and last_purchase_ts.
- The function is defined in the main.bakehouse schema.

In [0]:
%sql
-- Define a function to get high-value customers as a table-valued function. High value customers are defined as those who have spent more than the average amount spent by all customers in the last 14 days.
-- The function takes three arguments: franchise_id (INT), top_k and drop_window_days with default values of 10 and 14 respectively.
-- The function returns a table with three columns: customer_id, total_spent, and last_purchase_ts.
-- The function is defined in the main.bakehouse schema.
DROP FUNCTION IF EXISTS main.bakehouse.get_high_value_at_risk_customers;

CREATE OR REPLACE FUNCTION main.bakehouse.get_high_value_at_risk_customers(franchise_id INT)
RETURNS TABLE (
  customer_id BIGINT,
  total_spent DOUBLE,
  last_purchase_ts TIMESTAMP
)
LANGUAGE SQL
RETURN
WITH
  params AS (
    SELECT
      10  AS top_k,
      14  AS drop_window_days
  ),
  filtered_transactions AS (
    SELECT *
    FROM main.bakehouse.sales_transactions
    WHERE franchise_id = get_high_value_at_risk_customers.franchise_id
  ),
  customer_spend AS (
    SELECT
      t.customer_id,
      SUM(t.total_price) AS total_spent,
      MAX(t.date_time)   AS last_purchase_ts
    FROM filtered_transactions t
    GROUP BY t.customer_id
  ),
  dataset_last_date AS (
    SELECT DATE(MAX(date_time)) AS last_date
    FROM filtered_transactions
  ),
  avg_spent AS (
    SELECT AVG(total_spent) AS avg_total_spent
    FROM (
      SELECT
        customer_id,
        SUM(total_price) AS total_spent
      FROM filtered_transactions
      WHERE date_time >= DATE_SUB((SELECT last_date FROM dataset_last_date), (SELECT drop_window_days FROM params))
      GROUP BY customer_id
    )
  ),
  ranked AS (
    SELECT
      cs.customer_id,
      cs.total_spent,
      cs.last_purchase_ts,
      ROW_NUMBER() OVER (ORDER BY cs.total_spent DESC) AS rn
    FROM customer_spend cs
    CROSS JOIN dataset_last_date d
    CROSS JOIN params p
    CROSS JOIN avg_spent a
    WHERE cs.last_purchase_ts < DATE_SUB(d.last_date, p.drop_window_days)
      AND cs.total_spent > a.avg_total_spent
  )
SELECT
  customer_id,
  total_spent,
  last_purchase_ts
FROM ranked
CROSS JOIN params p
WHERE rn <= p.top_k;

### Test get_high_value_at_risk_customers

In [0]:
%sql
SELECT * FROM main.bakehouse.get_high_value_at_risk_customers(1);


## 2/ Get customer profile information
This Unity Catalog SQL table-valued function returns an enriched customer profile for a given `customer_id`. It reads directly from `main.bakehouse.sales_customers` and `main.bakehouse.sales_transactions`, so results always reflect the latest data.

The function:
- Looks up core customer attributes (name, contact info, address, demographics, franchise).  
- Computes the customer’s **age group** (`retiree`, `young adult`, or `adult`) from the current date and `birth_date`.  
- Analyzes all transactions for that customer to derive visit behavior and purchase preferences.

From the transaction history it calculates:
- Total number of visits.  
- Number of work‑week visit days, weekend visit days, and distinct visit days overall.  
- Average weekly transaction count and a **frequency label**:
  - `Frequent`: 2–7 transactions per week on average.  
  - `Normal`: 1–2 transactions per week on average.  
  - `Occasional`: fewer than 1 transaction per week.  
- Preferred **payment method** (card, mobile, cash) based on the most commonly used method.  
- **Most frequently purchased item** for the customer.  
- **Least frequently purchased item**, computed as the product with the lowest transaction count for that customer.  
- **Favorite visit time** bucket (`morning`, `lunch`, `afternoon`, `dinner`) based on when the customer most often visits.

Because all aggregations are done on the fly inside the function, you do not need to maintain any intermediate profile tables. Calling:

`SELECT * FROM main.bakehouse.get_customer_information(<customer_id>);`

In [0]:
%sql
--  prompt for coding assistant: 
--  From sales_transactions and sales_customers table, return a table with:
--  - age (computed from current date - birth_date): retiree, young adult (<30), adult
--  - total number of visits
--  - work week visitors, whole week visitors and weekend visitors
--  - Frequent : 2-7 transactions weekly, Normal: 1-2 transactions weekly, default : Occasional
--  - purchase type (card, mobile, cash)
--  - most frequently purchased item
--  - least frequently purchased item
--  - favorite visit time (morning, lunch, afternoon, dinner)

DROP FUNCTION IF EXISTS main.bakehouse.get_customer_information;
CREATE OR REPLACE FUNCTION main.bakehouse.get_customer_information(
  customer_id BIGINT
)
RETURNS TABLE (
  customer_id          BIGINT,
  first_name           STRING,
  last_name            STRING,
  email_address        STRING,
  phone_number         STRING,
  address              STRING,
  city                 STRING,
  state                STRING,
  country              STRING,
  continent            STRING,
  postal_zip_code      STRING,
  gender               STRING,
  birth_date           DATE,
  franchise_id         BIGINT,
  age_group            STRING,
  total_visits         BIGINT,
  work_week_visits     BIGINT,
  whole_week_visits    BIGINT,
  weekend_visits       BIGINT,
  frequency            STRING,
  purchase_type        STRING,
  most_frequent_item   STRING,
  least_frequent_item  STRING,
  favorite_visit_time  STRING
)
LANGUAGE SQL
RETURN
WITH base_customer AS (
  SELECT *
  FROM main.bakehouse.sales_customers c
  WHERE c.customer_id = get_customer_information.customer_id
),
tx AS (
  SELECT
    t.customer_id,
    t.product,
    t.payment_method,
    t.date_time,
    DATE(t.date_time)            AS date,
    dayofweek(DATE(t.date_time)) AS weekday,
    hour(t.date_time)            AS hour,
    CASE
      WHEN hour(t.date_time) BETWEEN 6 AND 10  THEN 'morning'
      WHEN hour(t.date_time) BETWEEN 11 AND 13 THEN 'lunch'
      WHEN hour(t.date_time) BETWEEN 14 AND 17 THEN 'afternoon'
      ELSE 'dinner'
    END AS visit_time
  FROM main.bakehouse.sales_transactions t
  WHERE t.customer_id = get_customer_information.customer_id
),
cust_with_age AS (
  SELECT
    bc.*,
    FLOOR(months_between(current_date(), bc.birth_date) / 12) AS age,
    CASE
      WHEN FLOOR(months_between(current_date(), bc.birth_date) / 12) >= 60 THEN 'retiree'
      WHEN FLOOR(months_between(current_date(), bc.birth_date) / 12) < 30  THEN 'young adult'
      ELSE 'adult'
    END AS age_group
  FROM base_customer bc
),
-- per-product counts per customer (for least frequent item)
item_counts AS (
  SELECT
    customer_id,
    product,
    COUNT(*) AS cnt
  FROM tx
  GROUP BY customer_id, product
),
least_item AS (
  SELECT
    customer_id,
    FIRST(product) AS least_frequent_item
  FROM (
    SELECT
      customer_id,
      product,
      cnt,
      ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY cnt ASC, product ASC
      ) AS rn
    FROM item_counts
  ) x
  WHERE rn = 1
  GROUP BY customer_id
),
-- per-customer aggregates
agg AS (
  SELECT
    t.customer_id,
    COUNT(*)                                              AS total_visits,
    COUNT(DISTINCT CASE WHEN weekday BETWEEN 2 AND 6 THEN date END) AS work_week_visits,
    COUNT(DISTINCT CASE WHEN weekday IN (1,7) THEN date END)        AS weekend_visits,
    COUNT(DISTINCT date)                                   AS whole_week_visits,
    MODE() WITHIN GROUP (ORDER BY payment_method)          AS purchase_type,
    MODE() WITHIN GROUP (ORDER BY product)                 AS most_frequent_item,
    MODE() WITHIN GROUP (ORDER BY visit_time)              AS favorite_visit_time,
    COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT weekofyear(date)), 0) AS avg_weekly_tx
  FROM tx t
  GROUP BY t.customer_id
),
agg_with_least AS (
  SELECT
    a.*,
    li.least_frequent_item
  FROM agg a
  LEFT JOIN least_item li
    ON a.customer_id = li.customer_id
),
freq AS (
  SELECT
    a.*,
    CASE
      WHEN avg_weekly_tx >= 2 THEN 'Frequent'
      WHEN avg_weekly_tx >= 1 THEN 'Normal'
      ELSE 'Occasional'
    END AS frequency
  FROM agg_with_least a
)
SELECT
  c.customer_id,
  c.first_name,
  c.last_name,
  c.email_address,
  c.phone_number,
  c.address,
  c.city,
  c.state,
  c.country,
  c.continent,
  c.postal_zip_code,
  c.gender,
  c.birth_date,
  c.franchise_id,
  c.age_group,
  COALESCE(f.total_visits,        0)            AS total_visits,
  COALESCE(f.work_week_visits,    0)            AS work_week_visits,
  COALESCE(f.whole_week_visits,   0)            AS whole_week_visits,
  COALESCE(f.weekend_visits,      0)            AS weekend_visits,
  COALESCE(f.frequency,           'Occasional') AS frequency,
  f.purchase_type,
  f.most_frequent_item,
  f.least_frequent_item,
  f.favorite_visit_time
FROM cust_with_age c
LEFT JOIN freq f
  ON c.customer_id = f.customer_id;


### Test get_customer_information

In [0]:
# Get customer information for all high-value at-risk customers
# Note: SQL table-valued functions with correlated subqueries cannot be used in LATERAL joins
# Using Python to call the function for each customer

# Get high-value at-risk customer IDs
high_value_customers = spark.sql("""
    SELECT customer_id 
    FROM main.bakehouse.get_high_value_at_risk_customers()
""").collect()

# Get customer information for each
customer_ids = [row.customer_id for row in high_value_customers]

if customer_ids:
    # Build a query that unions all customer information
    union_queries = [f"SELECT * FROM main.bakehouse.get_customer_information({cid})" for cid in customer_ids]
    full_query = " UNION ALL ".join(union_queries)
    
    result_df = spark.sql(full_query)
    display(result_df)
else:
    print("No high-value at-risk customers found")

In [0]:
df = spark.sql("SELECT * FROM main.bakehouse.get_customer_information(8)")
display(df)

## 3/ Get customer purchase history
This Unity Catalog SQL table-valued function returns the detailed purchase history for a single customer, identified by `customer_id`. It reads rows directly from `main.bakehouse.sales_transactions`, so the output always reflects the latest transactional data.

The function:
- Filters the `sales_transactions` table to only include transactions for the specified `customer_id`.
- Exposes each matching transaction as one row in the result table, preserving the original granularity of the source data.

For each transaction, the function returns:
- `transaction_id`: Unique identifier of the transaction.
- `franchise_id`: Store or franchise where the purchase was made.
- `date_time`: Timestamp of when the purchase occurred.
- `product`: Name or identifier of the purchased product.
- `quantity`: Number of units bought in that transaction line.
- `unit_price`: Price per unit at the time of purchase.
- `total_price`: Total amount for the line item (typically `quantity * unit_price`).
- `payment_method`: How the customer paid (for example, card, cash, mobile).
- `card_number`: Masked or stored representation of the payment card used, if applicable.

You can use this function from SQL or as an agent tool to:
- Inspect an individual customer’s order history.
- Drive downstream analytics (for example, spend over time, product preferences).
- Provide conversational explanations like “show me the last 10 purchases for this customer.”

Example usage:

`SELECT *
FROM main.bakehouse.get_customer_purchase_history('7')
ORDER BY date_time DESC;`

In [0]:
%sql
-- This is an example of a hosted function that will allow your agent to query a table and retrieve useful insights.
-- Select the catalog and schema you want to save the function to and run it.
-- Then go back to Playground an add the function by finding it with "Add existing tool"
CREATE OR REPLACE FUNCTION main.bakehouse.get_customer_purchase_history(customer_id STRING)
RETURNS TABLE (
  transaction_id STRING,
  franchise_id STRING,
  date_time TIMESTAMP,
  product STRING,
  quantity INT,
  unit_price DOUBLE,
  total_price DOUBLE,
  payment_method STRING,
  card_number STRING
)
LANGUAGE SQL
COMMENT 'Returns the purchase history for a given customer from sales_transactions.'
RETURN
  SELECT
    transaction_id,
    franchise_id,
    date_time,
    product,
    quantity,
    unit_price,
    total_price,
    payment_method,
    card_number
  FROM main.bakehouse.sales_transactions
  WHERE customer_id = get_customer_purchase_history.customer_id;

### Test get_customer_purchase_history

In [0]:
df = spark.sql("SELECT * FROM main.bakehouse.get_customer_purchase_history(7)")
display(df)

## 4/ List products
Let's define a function listing available products at a certain franchise so that the agent doesn't hallucinate products when generating content.

In [0]:
%sql
-- This is a hosted function that will return all products and price for a given franchise
CREATE OR REPLACE FUNCTION main.bakehouse.get_products(franchise INT)
RETURNS TABLE(product STRING, price FLOAT)
RETURN SELECT DISTINCT product, unit_price
FROM main.bakehouse.sales_transactions where franchise_id = franchise;

### Test get_products

In [0]:
df = spark.sql("SELECT * FROM main.bakehouse.get_products(1)")
display(df)

# Test using the chat in the playground
Let's go over to the AI Playground to see how we can use these functions and assemble our first Agent!

Open the [Playground](/ml/playground) and select the tools we created to test your agent!

<!-- <div style="float: right; width: 70%;">
  <img 
    src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/\
cross_demo_assets/AI_Agent_GIFs/AI_agent_function_selection.gif" 
    alt="Function Selection" 
    width="100%"
  >
</div> -->

### Location Guide

Your functions are organized in Unity Catalog using this structure:

#### Example Path:
`my_catalog.my_schema.my_awesome_function`. In our example, it will be `main.bakehouse.get*`

💡 Note: You can just type get in the search tool and click on the right function that come up.

## What's next: Evaluation

Our agent is now ready and leveraging our tools to properly answer our questions.

But how can we make sure it's working properly, and more importantly will still work well for future questions and modifications?

To do so, we need to build an Evaluation dataset and leverage MLFlow to automatically analyze our agent!
Open the [02-agent-eval/02.1_agent_evaluation]($../02-agent-eval/02.1_agent_evaluation) notebook to see how to deploy your agent using Langchain and run your first evaluations!